In [1]:
import requests
import pandas as pd

url = "https://data-api.binance.vision/api/v3/klines"

params = {
    "symbol": "BTCUSDT",
    "interval": "1h",
    "limit": 720
}

res = requests.get(url, params=params)
data = res.json()

df = pd.DataFrame(data, columns=[
    "time","open","high","low","close","volume",
    "close_time","qav","trades","taker_base","taker_quote","ignore"
])

df["close"] = df["close"].astype(float)
df = df[["time","close"]]

df.head()

,time,close
0,1775210400000,66883.08
1,1775214000000,67017.44
2,1775217600000,66722.16
3,1775221200000,66672.00
4,1775224800000,66929.81


In [2]:
import numpy as np

df["returns"] = np.log(df["close"] / df["close"].shift(1))
df = df.dropna()

In [7]:
def predict_range(returns, last_price):
    mu = returns.mean()
    sigma = returns.std()

    simulations = []

    for _ in range(10000):
        shock = np.random.standard_t(df=5)  # fat tails
        simulated_return = mu + sigma * shock
        simulated_price = last_price * np.exp(simulated_return)
        simulations.append(simulated_price)

    lower = np.percentile(simulations, 3)
    upper = np.percentile(simulations, 97)

    return lower, upper

In [8]:
predictions = []

for i in range(50, len(df)-1):  # start after some data
    past_returns = df["returns"].iloc[:i]
    last_price = df["close"].iloc[i]

    lower, upper = predict_range(past_returns, last_price)

    actual = df["close"].iloc[i+1]

    predictions.append({
        "lower": lower,
        "upper": upper,
        "actual": actual
    })

In [9]:
def evaluate(predictions):
    inside = 0
    widths = []

    for p in predictions:
        if p["lower"] <= p["actual"] <= p["upper"]:
            inside += 1
        widths.append(p["upper"] - p["lower"])

    coverage = inside / len(predictions)
    avg_width = np.mean(widths)

    return coverage, avg_width

coverage, avg_width = evaluate(predictions)

print("Coverage:", coverage)
print("Avg Width:", avg_width)

Coverage: 0.9655688622754491
Avg Width: 1444.8860366392194


In [10]:
import json

with open("backtest_results.jsonl", "w") as f:
    for p in predictions:
        f.write(json.dumps(p) + "\n")